#Bronce

In [10]:
import pandas as pd

path = "gs://final-julio-alvarez-bucket/bronce/raw/US_Accidents_sample_1k.csv"

df = pd.read_csv(
    path,
    sep=",",
    quotechar='"',
    engine="python",
    encoding="utf-8"
)


In [11]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[()]", "", regex=True)
      .str.replace("%", "pct")
      .str.replace("/", "_")
)


In [12]:
# Eliminar filas completamente vacías
df = df.dropna(how="all")

# Forzar booleanos a True/False
bool_cols = [
    "amenity","bump","crossing","give_way","junction","no_exit",
    "railway","roundabout","station","stop","traffic_calming",
    "traffic_signal","turning_loop"
]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)


In [13]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.head(2)


Filas: 1000
Columnas: 50


,id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distancemi,...,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight,hour,dayofweek,month,hour
0,A-7182628,Source1,1,2020-04-17 09:29:30,2020-04-17 10:29:30,26.706900,-80.11936,26.706900,-80.119360,0.000,...,True,False,Day,Day,Day,Day,9.0,4.0,4.0,9.0
1,A-5404588,Source1,2,NaN,NaN,38.781024,-121.26582,38.780377,-121.265815,0.045,...,False,False,Day,Day,Day,Day,NaN,NaN,NaN,NaN


In [15]:
# ============================================
# 1. NORMALIZACIÓN ESTRUCTURAL DEL DATAFRAME
# ============================================

# Eliminar columnas duplicadas (ej. 'hour')
df = df.loc[:, ~df.columns.duplicated()]

# Resetear índice para evitar errores internos de pandas
df = df.reset_index(drop=True)

# Validaciones obligatorias
assert df.columns.is_unique, "Error: existen columnas duplicadas"
assert df.index.is_monotonic_increasing, "Error: índice no válido"

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])


# ============================================
# 2. CARGA A BIGQUERY – CAPA BRONCE
# ============================================

from google.cloud import bigquery

PROJECT_ID = "final-julio-alvarez"
TABLE_ID = f"{PROJECT_ID}.bronce.accidents_raw"

client = bigquery.Client(project=PROJECT_ID)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = client.load_table_from_dataframe(
    df,
    TABLE_ID,
    job_config=job_config
)

load_job.result()

print("Carga a BRONCE completada correctamente.")



Filas: 1000
Columnas: 49
Carga a BRONCE completada correctamente.


In [20]:
from google.cloud import bigquery

client = bigquery.Client(project="final-julio-alvarez")

table_id = "final-julio-alvarez.bronce.accidents_raw"

table = client.get_table(table_id)

print("Columnas en BRONCE:\n")

for i, field in enumerate(table.schema, start=1):
    print(f"{i:02d}. {field.name} ({field.field_type})")


Columnas en BRONCE:

01. id (STRING)
02. source (STRING)
03. severity (INTEGER)
04. start_time (STRING)
05. end_time (STRING)
06. start_lat (FLOAT)
07. start_lng (FLOAT)
08. end_lat (FLOAT)
09. end_lng (FLOAT)
10. distancemi (FLOAT)
11. description (STRING)
12. street (STRING)
13. city (STRING)
14. county (STRING)
15. state (STRING)
16. zipcode (STRING)
17. country (STRING)
18. timezone (STRING)
19. airport_code (STRING)
20. weather_timestamp (STRING)
21. temperaturef (FLOAT)
22. wind_chillf (FLOAT)
23. humiditypct (FLOAT)
24. pressurein (FLOAT)
25. visibilitymi (FLOAT)
26. wind_direction (STRING)
27. wind_speedmph (FLOAT)
28. precipitationin (FLOAT)
29. weather_condition (STRING)
30. amenity (BOOLEAN)
31. bump (BOOLEAN)
32. crossing (BOOLEAN)
33. give_way (BOOLEAN)
34. junction (BOOLEAN)
35. no_exit (BOOLEAN)
36. railway (BOOLEAN)
37. roundabout (BOOLEAN)
38. station (BOOLEAN)
39. stop (BOOLEAN)
40. traffic_calming (BOOLEAN)
41. traffic_signal (BOOLEAN)
42. turning_loop (BOOLEAN)
43. 

#Plata

In [21]:
query_plata = """
CREATE OR REPLACE TABLE `final-julio-alvarez.plata.accidents_clean` AS
SELECT
  id,
  source,
  severity,

  -- Fechas
  PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S', start_time) AS start_ts,
  PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S', end_time) AS end_ts,

  -- Duración en minutos
  TIMESTAMP_DIFF(
    PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S', end_time),
    PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S', start_time),
    MINUTE
  ) AS duration_min,

  -- Ubicación
  start_lat,
  start_lng,
  city,
  county,
  state,
  zipcode,
  country,
  timezone,

  -- Clima
  weather_condition,
  temperaturef,
  humiditypct,
  pressurein,
  visibilitymi,
  wind_speedmph,
  precipitationin,

  -- Infraestructura vial (flags)
  amenity,
  bump,
  crossing,
  give_way,
  junction,
  no_exit,
  railway,
  roundabout,
  station,
  stop,
  traffic_calming,
  traffic_signal,
  turning_loop,

  -- Tiempo
  CAST(hour AS INT64) AS hour,
  CAST(dayofweek AS INT64) AS day_of_week,
  CAST(month AS INT64) AS month,
  sunrise_sunset,
  civil_twilight,
  nautical_twilight,
  astronomical_twilight

FROM `final-julio-alvarez.bronce.accidents_raw`
WHERE start_time IS NOT NULL
  AND end_time IS NOT NULL
"""
client.query(query_plata).result()

print("PLATA creada correctamente")


PLATA creada correctamente


#Oro

In [22]:
query_dim_tiempo = """
CREATE OR REPLACE TABLE `final-julio-alvarez.oro.dim_tiempo` AS
SELECT DISTINCT
  start_ts AS fecha_hora,
  EXTRACT(HOUR FROM start_ts) AS hora,
  EXTRACT(DAYOFWEEK FROM start_ts) AS dia_semana,
  EXTRACT(MONTH FROM start_ts) AS mes
FROM `final-julio-alvarez.plata.accidents_clean`
"""
client.query(query_dim_tiempo).result()


In [23]:
query_dim_ubicacion = """
CREATE OR REPLACE TABLE `final-julio-alvarez.oro.dim_ubicacion` AS
SELECT DISTINCT
  city,
  county,
  state,
  zipcode,
  country,
  start_lat,
  start_lng
FROM `final-julio-alvarez.plata.accidents_clean`
"""
client.query(query_dim_ubicacion).result()


In [24]:
query_dim_clima = """
CREATE OR REPLACE TABLE `final-julio-alvarez.oro.dim_clima` AS
SELECT DISTINCT
  weather_condition,
  temperaturef,
  humiditypct,
  pressurein,
  visibilitymi,
  wind_speedmph,
  precipitationin
FROM `final-julio-alvarez.plata.accidents_clean`
"""
client.query(query_dim_clima).result()


In [25]:
query_fact = """
CREATE OR REPLACE TABLE `final-julio-alvarez.oro.fact_accidentes` AS
SELECT
  a.id,

  -- FK naturales
  a.start_ts AS fecha_hora,
  a.city,
  a.state,
  a.weather_condition,

  -- Métricas
  a.severity,
  a.duration_min,

  -- Indicadores
  a.traffic_signal,
  a.junction,
  a.crossing

FROM `final-julio-alvarez.plata.accidents_clean` a
"""
client.query(query_fact).result()

print("MODELO ESTRELLA (ORO) creado correctamente")


MODELO ESTRELLA (ORO) creado correctamente
